## Inferencia sobre validacion - metricas CV (STL simetria)

Calcula las metricas de los 5 folds de cada modelo sobre su conjunto de **validacion** retenido (reconstruido con el mismo `StratifiedKFold(random_state=42)` del entrenamiento, estratificado por la etiqueta de simetria). Carga los `best_weights` guardados y predice; no reentrena.

Notebook autocontenido. Solo hay que poner las carpetas de experimento reales en `MODELS`.

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score, roc_auc_score,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"   

train_csv  = "/home/marc/MARIADELMAR_EXPERIMENTS/onehot_files/train_onehot.csv"
val_csv    = "/home/marc/MARIADELMAR_EXPERIMENTS/onehot_files/val_onehot.csv"
images_dir = "/home/marc/MARIADELMAR_EXPERIMENTS/dataverse_files/images"
sym_csv    = "/home/marc/MARIADELMAR_EXPERIMENTS/ham10000_shape_symmetry_ALL.csv"

NUM_CLASSES = 3
N_FOLDS     = 5
class_names = ["2_ejes", "1_eje", "asimetrica"]

# Conjunto train+val con la etiqueta de simetria, mismo split que en entrenamiento
df_trainval = pd.concat([pd.read_csv(train_csv), pd.read_csv(val_csv)], ignore_index=True)
df_sym = pd.read_csv(sym_csv).rename(columns={"image": "image_id"})[["image_id", "shape_symmetry"]]
df_trainval = df_trainval.merge(df_sym, on="image_id", how="inner").reset_index(drop=True)
df_trainval["filepath"] = df_trainval["image_id"].apply(lambda x: os.path.join(images_dir, f"{x}.jpg"))
y_trainval_int = df_trainval["shape_symmetry"].values.astype(int)

skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(skf.split(np.zeros(len(df_trainval)), y_trainval_int))


def make_ds(filepaths, img_size, preprocess_fn, batch=32):
    def _load(fp):
        img = tf.io.read_file(fp)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)
        return preprocess_fn(img)
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(_load, num_parallel_calls=15)
    return ds.batch(batch).prefetch(50)


def compute_all_metrics(y_true_int, proba):
    pred = np.argmax(proba, axis=1)
    m = {
        "sym_acc":             float((y_true_int == pred).mean()),
        "sym_balanced_acc":    float(balanced_accuracy_score(y_true_int, pred)),
        "sym_precision_macro": float(precision_score(y_true_int, pred, average="macro", zero_division=0)),
        "sym_recall_macro":    float(recall_score(y_true_int, pred, average="macro", zero_division=0)),
        "sym_f1_macro":        float(f1_score(y_true_int, pred, average="macro", zero_division=0)),
        "sym_f1_weighted":     float(f1_score(y_true_int, pred, average="weighted", zero_division=0)),
        "sym_kappa":           float(cohen_kappa_score(y_true_int, pred)),
    }
    try:
        m["sym_auc_macro"] = float(roc_auc_score(np.eye(NUM_CLASSES)[y_true_int],
                                   proba, multi_class="ovr", average="macro"))
    except Exception:
        m["sym_auc_macro"] = float("nan")
    return m

2026-06-23 10:40:48.421100: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1


In [2]:
preprocess_vgg16       = lambda img: tf.keras.applications.vgg16.preprocess_input(img)
preprocess_resnet50    = lambda img: tf.keras.applications.resnet.preprocess_input(img)
preprocess_mobilenetv2 = lambda img: tf.keras.applications.mobilenet_v2.preprocess_input(img)
preprocess_densenet169 = lambda img: tf.keras.applications.densenet.preprocess_input(img)
preprocess_unet        = lambda img: img / 255.0

In [3]:
def _tl_head(base, img_size, arch):
    base.trainable = False
    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_symmetry")(x)
    return tf.keras.Model(inputs, out, name=f"STL_{arch}_symmetry")

def build_vgg16(img_size):
    return _tl_head(tf.keras.applications.VGG16(include_top=False, weights="imagenet",
                    input_shape=(img_size[0], img_size[1], 3)), img_size, "VGG16")
def build_resnet50(img_size):
    return _tl_head(tf.keras.applications.ResNet50(include_top=False, weights="imagenet",
                    input_shape=(img_size[0], img_size[1], 3)), img_size, "ResNet50")
def build_mobilenetv2(img_size):
    return _tl_head(tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet",
                    input_shape=(img_size[0], img_size[1], 3)), img_size, "MobileNetV2")
def build_densenet169(img_size):
    return _tl_head(tf.keras.applications.DenseNet169(include_top=False, weights="imagenet",
                    input_shape=(img_size[0], img_size[1], 3)), img_size, "DenseNet169")

def build_unet(img_size):
    # U-Net autoencoder simetria, 2 salidas [reconstruccion, head_sym]. proba[1] en el bucle.
    def enc(inp, f):
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(inp)
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        return x, tf.keras.layers.MaxPool2D(2,2)(x)
    def dec(inp, skip, f):
        u = tf.keras.layers.UpSampling2D(2)(inp)
        x = tf.keras.layers.Conv2D(f,2,activation="relu",padding="same",kernel_initializer="he_normal")(u)
        x = tf.keras.layers.concatenate([skip, x], axis=3)
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Conv2D(f,3,activation="relu",padding="same",kernel_initializer="he_normal")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        return x
    inputs = tf.keras.layers.Input(shape=(img_size[0], img_size[1], 3), name="input_image")
    c1,p1 = enc(inputs,64); c2,p2 = enc(p1,128); c3,p3 = enc(p2,256); c4,p4 = enc(p3,512)
    b1 = tf.keras.layers.Conv2D(1024,3,activation="relu",padding="same",kernel_initializer="he_normal")(p4)
    b1 = tf.keras.layers.BatchNormalization()(b1)
    b1 = tf.keras.layers.Conv2D(1024,3,activation="relu",padding="same",kernel_initializer="he_normal",name="bottleneck_conv")(b1)
    b1 = tf.keras.layers.BatchNormalization(name="bottleneck_bn")(b1)
    e1 = dec(b1,c4,512); e2 = dec(e1,c3,256); e3 = dec(e2,c2,128); e4 = dec(e3,c1,64)
    fr = tf.keras.layers.Conv2D(64,3,activation="relu",padding="same",kernel_initializer="he_normal")(e4)
    fr = tf.keras.layers.BatchNormalization()(fr)
    recon = tf.keras.layers.Conv2D(3,1,activation="relu",padding="same",name="reconstruction_output")(fr)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(b1)
    x = tf.keras.layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="head_sym")(x)
    return tf.keras.Model(inputs, [recon, out], name="STL_UNet_symmetry_autoencoder")

In [4]:
BASE_SYM = Path("/home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos")


MODELS = [
    {"name":"VGG16",       "img_size":(224,224), "preprocess":preprocess_vgg16,
     "build_fn":build_vgg16,       "exp_dir":BASE_SYM/"VGG16_STL"/"exp_2026-04-26_19-11_symmetry_CUT7_5fold"},
    {"name":"ResNet50",    "img_size":(224,224), "preprocess":preprocess_resnet50,
     "build_fn":build_resnet50,    "exp_dir":BASE_SYM/"ResNet50_STL"/"exp_2026-04-26_20-25_symmetry_CUT39_5fold"},
    {"name":"MobileNetV2", "img_size":(224,224), "preprocess":preprocess_mobilenetv2,
     "build_fn":build_mobilenetv2, "exp_dir":BASE_SYM/"MobileNetV2_STL"/"exp_2026-04-26_18-08_symmetry_CUT0_5fold"},
    {"name":"DenseNet169", "img_size":(224,224), "preprocess":preprocess_densenet169,
     "build_fn":build_densenet169, "exp_dir":BASE_SYM/"DenseNet_STL"/"exp_2026-04-26_11-14_symmetry_CUT0_5fold"},
    {"name":"UNet",        "img_size":(256,256), "preprocess":preprocess_unet,
     "build_fn":build_unet,        "exp_dir":BASE_SYM/"UNet_STL"/"exp_2026-06-22_20-34_symmetry_5fold"},
]

print("Verificando rutas de experimentos:")
for m in MODELS:
    print(f"  [{'OK' if m['exp_dir'].exists() else 'FALTA'}] {m['name']:<14} {m['exp_dir']}")

Verificando rutas de experimentos:
  [OK] VGG16          /home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos/VGG16_STL/exp_2026-04-26_19-11_symmetry_CUT7_5fold
  [OK] ResNet50       /home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos/ResNet50_STL/exp_2026-04-26_20-25_symmetry_CUT39_5fold
  [OK] MobileNetV2    /home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos/MobileNetV2_STL/exp_2026-04-26_18-08_symmetry_CUT0_5fold
  [OK] DenseNet169    /home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos/DenseNet_STL/exp_2026-04-26_11-14_symmetry_CUT0_5fold
  [OK] UNet           /home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos/UNet_STL/exp_2026-06-22_20-34_symmetry_5fold


In [5]:
for cfg in MODELS:
    name    = cfg["name"]
    exp_dir = Path(cfg["exp_dir"])
    print(name)

    rows = []
    for fold_idx, (train_idx, val_idx) in enumerate(splits, 1):
        fold_dir = exp_dir / f"fold_{fold_idx}"
        if not (fold_dir / "best_weights.index").exists():
            print(f"  fold {fold_idx}: sin pesos, se salta")
            continue

        df_vl    = df_trainval.iloc[val_idx].reset_index(drop=True)
        y_vl_int = df_vl["shape_symmetry"].values.astype(int)
        val_ds   = make_ds(df_vl["filepath"].values, cfg["img_size"], cfg["preprocess"])

        tf.keras.backend.clear_session()
        built = cfg["build_fn"](cfg["img_size"])
        model = built[0] if isinstance(built, tuple) else built
        for xb in val_ds.take(1):
            _ = model(xb[:1], training=False)
        model.load_weights(str(fold_dir / "best_weights")).expect_partial()

        proba = model.predict(val_ds, verbose=0)
        if isinstance(proba, (list, tuple)):   # U-Net autoencoder devuelve [recon, clasif]
            proba = proba[1]

        m = compute_all_metrics(y_vl_int, proba)
        rows.append({"fold": fold_idx, **m})
        print(f"  fold {fold_idx}: f1_macro={m['sym_f1_macro']:.4f}  bal_acc={m['sym_balanced_acc']:.4f}")

    df_val = pd.DataFrame(rows)
    df_val.to_csv(exp_dir / "all_folds_VAL_metrics.csv", index=False)
    num  = [c for c in df_val.columns if c != "fold"]
    summ = df_val[num].agg(["mean", "median", "std"]).T
    summ.columns = ["mean", "median", "std"]
    summ.to_csv(exp_dir / "summary_VAL_mean_median_std.csv")
    print(f"  rendimiento esperado (validacion) - {name}:")
    print(summ[["mean", "std"]].round(4))
    print(f"  guardado en: {exp_dir}")
    print()

VGG16


2026-06-23 10:40:49.950181: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcuda.so.1
2026-06-23 10:40:49.982891: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1716] Found device 0 with properties: 
pciBusID: 0000:06:00.0 name: GeForce GTX 1080 Ti computeCapability: 6.1
coreClock: 1.582GHz coreCount: 28 deviceMemorySize: 10.92GiB deviceMemoryBandwidth: 451.17GiB/s
2026-06-23 10:40:49.982950: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1
2026-06-23 10:40:49.985903: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcublas.so.10
2026-06-23 10:40:49.988493: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcufft.so.10
2026-06-23 10:40:49.988976: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcurand.so

  fold 1: f1_macro=0.5002  bal_acc=0.5057
  fold 2: f1_macro=0.5103  bal_acc=0.5465
  fold 3: f1_macro=0.4518  bal_acc=0.5099
  fold 4: f1_macro=0.5385  bal_acc=0.5578
  fold 5: f1_macro=0.4726  bal_acc=0.4838
  rendimiento esperado (validacion) — VGG16:
                       mean     std
sym_acc              0.6194  0.0254
sym_balanced_acc     0.5208  0.0306
sym_precision_macro  0.5066  0.0311
sym_recall_macro     0.5208  0.0306
sym_f1_macro         0.4947  0.0336
sym_f1_weighted      0.5975  0.0278
sym_kappa            0.2729  0.0424
sym_auc_macro        0.7540  0.0238
  guardado en: /home/marc/MARIADELMAR_EXPERIMENTS/STL_Symmetry_experimentos/VGG16_STL/exp_2026-04-26_19-11_symmetry_CUT7_5fold

ResNet50
  fold 1: f1_macro=0.5880  bal_acc=0.5987
  fold 2: f1_macro=0.5569  bal_acc=0.5842
  fold 3: f1_macro=0.5084  bal_acc=0.5858
  fold 4: f1_macro=0.5379  bal_acc=0.5779
  fold 5: f1_macro=0.5481  bal_acc=0.5686
  rendimiento esperado (validacion) — ResNet50:
                       mea

In [7]:
df_val = pd.read_csv(val_csv_f)
df_test = pd.read_csv(test_csv_f)

print("VAL columns:")
print(df_val.columns.tolist())

print("\nTEST columns:")
print(df_test.columns.tolist())

VAL columns:
['fold', 'sym_acc', 'sym_balanced_acc', 'sym_precision_macro', 'sym_recall_macro', 'sym_f1_macro', 'sym_f1_weighted', 'sym_kappa', 'sym_auc_macro']

TEST columns:
['fold', 'time_warmup_s', 'time_ft_s', 'time_total_s', 'time_predict_s', 'epochs_warmup', 'epochs_ft', 'len_warmup', 'symmetry_acc', 'symmetry_balanced_acc', 'symmetry_precision_macro', 'symmetry_recall_macro', 'symmetry_f1_macro', 'symmetry_f1_weighted', 'symmetry_kappa', 'symmetry_auc_macro']


In [8]:
# Comparacion val vs test (F1 macro de simetria) por modelo
for cfg in MODELS:
    exp_dir = Path(cfg["exp_dir"])
    val_csv_f  = exp_dir / "all_folds_VAL_metrics.csv"
    test_csv_f = exp_dir / "all_folds_metrics.csv"
    if not (val_csv_f.exists() and test_csv_f.exists()):
        print(f"{cfg['name']}: falta algun CSV, se salta"); continue
    f1_val  = pd.read_csv(val_csv_f)["sym_f1_macro"].mean()
    f1_test = pd.read_csv(test_csv_f)["symmetry_f1_macro"].mean()
    print(f"{cfg['name']:<14} f1_val={f1_val:.4f}  f1_test={f1_test:.4f}  brecha={f1_val - f1_test:+.4f}")

VGG16          f1_val=0.4947  f1_test=0.5029  brecha=-0.0082
ResNet50       f1_val=0.5478  f1_test=0.5504  brecha=-0.0026
MobileNetV2    f1_val=0.5447  f1_test=0.5455  brecha=-0.0008
DenseNet169    f1_val=0.5768  f1_test=0.5800  brecha=-0.0032
UNet           f1_val=0.4128  f1_test=0.4104  brecha=+0.0024
